# Sephora Product Intelligence — Feature Engineering

This notebook transforms clean product and review data into rich analytical 
features that will feed our machine learning model. We build 5 groups of 
features that capture review quality, price value, engagement, and ingredient 
quality — giving the model a complete picture of each product beyond just its 
star rating.

## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)
plt.style.use("seaborn-v0_8-whitegrid")

print("Libraries loaded")

Libraries loaded


## 2. Load Cleaned Data
Loading from data/processed/ — the output of our cleaning notebook.
We never touch the raw data again from this point forward.

In [2]:
products = pd.read_csv("../data/processed/products_clean.csv")
reviews = pd.read_csv("../data/processed/reviews_clean.csv")

print(f"Products loaded: {products.shape}")
print(f"Reviews loaded: {reviews.shape}")
print(f"\nProduct columns: {list(products.columns)}")
print(f"\nReview columns: {list(reviews.columns)}")

Products loaded: (1952, 22)
Reviews loaded: (980344, 19)

Product columns: ['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count', 'rating', 'reviews', 'size', 'variation_type', 'variation_value', 'ingredients', 'price_usd', 'limited_edition', 'new', 'online_only', 'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category', 'secondary_category', 'tertiary_category', 'child_count']

Review columns: ['author_id', 'rating', 'is_recommended', 'helpfulness', 'total_feedback_count', 'total_neg_feedback_count', 'total_pos_feedback_count', 'submission_time', 'review_text', 'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color', 'product_id', 'product_name', 'brand_name', 'price_usd', 'review_year']


## 3. Group 1 — Review Aggregation Metrics

We aggregate 980,344 individual reviews down to one row per product.
This gives us signals about how customers actually feel about each product.

- avg_rating: mean star rating — our ML target variable
- review_count: number of reviews — more = more trustworthy
- five_star_share: proportion of 5-star reviews — measures enthusiasm
- one_star_share: proportion of 1-star reviews — measures dissatisfaction
- recommendation_rate: % who explicitly recommended the product
- avg_helpfulness: how trusted the reviews are by other users

In [3]:
review_metrics = reviews.groupby("product_id").agg(
    avg_rating=("rating", "mean"),
    review_count=("rating", "count"),
    five_star_share=("rating", lambda x: (x == 5).mean()),
    one_star_share=("rating", lambda x: (x == 1).mean()),
    recommendation_rate=("is_recommended", "mean"),
    avg_helpfulness=("helpfulness", "mean")
).reset_index()

print(f"Review metrics shape: {review_metrics.shape}")
print(f"\nSample:")
print(review_metrics.head())
print(f"\nNull check:")
print(review_metrics.isnull().sum())

Review metrics shape: (1915, 7)

Sample:
  product_id  avg_rating  review_count  five_star_share  one_star_share  \
0    P107306       4.032           253            0.565           0.107   
1    P114902       4.420          1529            0.688           0.046   
2     P12045       4.443          1687            0.686           0.034   
3    P122651       4.515           200            0.725           0.040   
4    P122661       4.532           810            0.717           0.026   

   recommendation_rate  avg_helpfulness  
0                0.909            0.665  
1                0.969            0.393  
2                0.923            0.402  
3                0.975            0.457  
4                0.978            0.288  

Null check:
product_id             0
avg_rating             0
review_count           0
five_star_share        0
one_star_share         0
recommendation_rate    0
avg_helpfulness        0
dtype: int64


## 4. Group 2 — Price & Value Metrics

Price alone doesn't tell us much — $50 is cheap for a serum but expensive 
for a cleanser. We need price in context.

- price_rank_pct: percentile rank within skincare — 0.9 means more expensive 
  than 90% of all skincare products
- rating_per_dollar: value efficiency — how much rating per dollar spent
- price_tier: human readable label for easy filtering

In [4]:
# Work on a copy of products
products_features = products.copy()

# Price percentile rank within full skincare category
products_features["price_rank_pct"] = products_features["price_usd"].rank(pct=True)

# Rating per dollar — we'll recalculate after merging with review metrics
# For now use the site rating as placeholder
products_features["rating_per_dollar"] = products_features["rating"] / products_features["price_usd"]

# Price tier labels
products_features["price_tier"] = pd.cut(
    products_features["price_usd"],
    bins=[0, 25, 75, 150, 2000],
    labels=["Budget", "Mid", "Premium", "Luxury"]
)

print("Price tier distribution:")
print(products_features["price_tier"].value_counts())
print(f"\nPrice rank sample:")
print(products_features[["product_name", "price_usd", "price_rank_pct", "price_tier"]].head(10))

Price tier distribution:
price_tier
Mid        1184
Budget      387
Premium     300
Luxury       81
Name: count, dtype: int64

Price rank sample:
                                        product_name  price_usd  \
0               GENIUS Sleeping Collagen Moisturizer     98.000   
1                       GENIUS Liquid Collagen Serum    115.000   
2            Triple Algae Eye Renewal Balm Eye Cream     68.000   
3               GENIUS Liquid Collagen Lip Treatment     29.000   
4  SUBLIME DEFENSE Ultra Lightweight UV Defense F...     28.000   
5                   GENIUS Ultimate Anti-Aging Cream    112.000   
6        GENIUS Ultimate Anti-Aging Melting Cleanser     38.000   
7                       Gentle Rejuvenating Cleanser     28.000   
8                  Advanced Anti-Aging Repairing Oil     82.000   
9                        Overnight Restorative Cream     94.000   

   price_rank_pct price_tier  
0           0.891    Premium  
1           0.921    Premium  
2           0.726      

## 5. Group 3 — Engagement Metrics

Sephora users can love a product without reviewing it.
This gives us a popularity signal completely separate from reviews.

- engagement_quality: loves divided by review count — high ratio means 
  people love the product silently, indicating a cult following
- review_volume_rank: puts review volume in context across all products

In [5]:
# Merge review metrics into products first
products_features = products_features.merge(review_metrics, on="product_id", how="left")

# Now calculate engagement features using actual review_count
products_features["engagement_quality"] = (
    products_features["loves_count"] / (products_features["review_count"] + 1)
)

products_features["review_volume_rank"] = products_features["review_count"].rank(pct=True)

# Recalculate rating_per_dollar using actual avg_rating from reviews
products_features["rating_per_dollar"] = (
    products_features["avg_rating"] / products_features["price_usd"]
)

print(f"Products with review metrics merged: {products_features.shape}")
print(f"\nProducts WITH review data: {products_features['avg_rating'].notna().sum()}")
print(f"Products WITHOUT review data: {products_features['avg_rating'].isna().sum()}")
print(f"\nSample engagement metrics:")
print(products_features[["product_name", "loves_count", "review_count", "engagement_quality"]].head(10))

Products with review metrics merged: (1952, 33)

Products WITH review data: 1915
Products WITHOUT review data: 37

Sample engagement metrics:
                                        product_name  loves_count  \
0               GENIUS Sleeping Collagen Moisturizer        33910   
1                       GENIUS Liquid Collagen Serum        67870   
2            Triple Algae Eye Renewal Balm Eye Cream        17890   
3               GENIUS Liquid Collagen Lip Treatment        44448   
4  SUBLIME DEFENSE Ultra Lightweight UV Defense F...        27278   
5                   GENIUS Ultimate Anti-Aging Cream        19733   
6        GENIUS Ultimate Anti-Aging Melting Cleanser         9314   
7                       Gentle Rejuvenating Cleanser         7681   
8                  Advanced Anti-Aging Repairing Oil        10676   
9                        Overnight Restorative Cream        10578   

   review_count  engagement_quality  
0      1321.000              25.651  
1      1159.000       

## 6. Group 4 — Ingredient Scoring

This is the most unique part of our analysis. We parse the raw ingredient 
string for each product into individual ingredients, then check each one 
against a predefined list of known beneficial and problematic ingredients.

The matching uses partial string matching — so "retinyl" matches both 
"retinyl palmitate" and "retinyl acetate". Each ingredient is only 
counted once per product to avoid double counting.

- ingredient_count: total number of ingredients in the formula
- good_ingredient_count: count of beneficial ingredients found
- bad_ingredient_count: count of problematic ingredients found
- ingredient_score: (good - bad) / total — normalized net quality score
- good_bad_ratio: good / (bad + 1) — ratio of good to bad
- powerhouse_flag: 1 if product contains 3+ hero ingredients
- value_score: combines ingredient quality, rating, and price

In [6]:
# Define good and bad ingredient lists
good_ingredients = [
    "hyaluronic acid", "retinol", "retinyl", "niacinamide", "vitamin c",
    "ascorbic acid", "peptide", "ceramide", "glycerin", "squalane",
    "collagen", "vitamin e", "tocopherol", "salicylic acid", "glycolic acid",
    "lactic acid", "kojic acid", "azelaic acid", "resveratrol", "ferulic acid",
    "zinc", "allantoin", "panthenol", "centella", "madecassoside",
    "tranexamic acid", "arbutin", "adenosine", "bakuchiol", "coenzyme q10",
    "ubiquinone", "green tea", "epigallocatechin", "caffeine", "licorice",
    "aloe vera", "aloe barbadensis", "jojoba", "rosehip", "sea buckthorn",
    "vitamin b3", "vitamin b5", "alpha arbutin", "lipo hydroxy acid",
    "phytic acid", "mandelic acid"
]

bad_ingredients = [
    "paraben", "methylparaben", "propylparaben", "butylparaben",
    "sodium lauryl sulfate", "sodium laureth sulfate", "sls", "sles",
    "formaldehyde", "dmdm hydantoin", "imidazolidinyl urea",
    "diazolidinyl urea", "quaternium-15", "bronopol",
    "artificial fragrance", "synthetic fragrance",
    "phthalate", "dibutyl phthalate", "diethyl phthalate",
    "oxybenzone", "octinoxate", "homosalate", "octisalate",
    "polyethylene glycol", "propylene glycol",
    "butylated hydroxyanisole", "bha", "bht",
    "triclosan", "triclocarban",
    "talc", "mineral oil", "petrolatum",
    "alcohol denat", "sd alcohol", "isopropyl alcohol"
]

print(f"Good ingredients defined: {len(good_ingredients)}")
print(f"Bad ingredients defined: {len(bad_ingredients)}")

Good ingredients defined: 46
Bad ingredients defined: 36


In [7]:
# Parse ingredients string into a clean lowercase string
def parse_ingredients(ingredient_str):
    if pd.isna(ingredient_str):
        return ""
    # Remove brackets, quotes, list formatting
    cleaned = str(ingredient_str).lower()
    cleaned = cleaned.replace("[", "").replace("]", "")
    cleaned = cleaned.replace("'", "").replace('"', "")
    return cleaned

# Count good ingredients — partial match, count each only once
def count_good(ingredient_str):
    cleaned = parse_ingredients(ingredient_str)
    if not cleaned:
        return 0
    count = 0
    for good in good_ingredients:
        if good in cleaned:
            count += 1
    return count

# Count bad ingredients — partial match, count each only once
def count_bad(ingredient_str):
    cleaned = parse_ingredients(ingredient_str)
    if not cleaned:
        return 0
    count = 0
    for bad in bad_ingredients:
        if bad in cleaned:
            count += 1
    return count

# Count total ingredients
def count_total(ingredient_str):
    cleaned = parse_ingredients(ingredient_str)
    if not cleaned:
        return 0
    # Split by comma to count individual ingredients
    parts = [p.strip() for p in cleaned.split(",") if p.strip()]
    return len(parts)

# Apply to all products
products_features["ingredient_count"] = products_features["ingredients"].apply(count_total)
products_features["good_ingredient_count"] = products_features["ingredients"].apply(count_good)
products_features["bad_ingredient_count"] = products_features["ingredients"].apply(count_bad)

print("Ingredient counts applied")
print(f"\nSample:")
print(products_features[["product_name", "ingredient_count", "good_ingredient_count", "bad_ingredient_count"]].head(10))

Ingredient counts applied

Sample:
                                        product_name  ingredient_count  \
0               GENIUS Sleeping Collagen Moisturizer                43   
1                       GENIUS Liquid Collagen Serum                40   
2            Triple Algae Eye Renewal Balm Eye Cream                64   
3               GENIUS Liquid Collagen Lip Treatment                38   
4  SUBLIME DEFENSE Ultra Lightweight UV Defense F...                29   
5                   GENIUS Ultimate Anti-Aging Cream                39   
6        GENIUS Ultimate Anti-Aging Melting Cleanser                20   
7                       Gentle Rejuvenating Cleanser                30   
8                  Advanced Anti-Aging Repairing Oil                12   
9                        Overnight Restorative Cream                41   

   good_ingredient_count  bad_ingredient_count  
0                      4                     0  
1                      6                     1  
2  

In [8]:
# Calculate derived ingredient features
products_features["ingredient_score"] = (
    (products_features["good_ingredient_count"] - products_features["bad_ingredient_count"]) /
    (products_features["ingredient_count"] + 1)
)

products_features["good_bad_ratio"] = (
    products_features["good_ingredient_count"] /
    (products_features["bad_ingredient_count"] + 1)
)

# Powerhouse flag — 3 or more good ingredients
products_features["powerhouse_flag"] = (
    products_features["good_ingredient_count"] >= 3
).astype(int)

# Value score — good ingredients × rating / (price × bad ingredients)
products_features["value_score"] = (
    (products_features["good_ingredient_count"] * products_features["avg_rating"]) /
    (products_features["price_usd"] * (products_features["bad_ingredient_count"] + 1))
)

print("Derived ingredient features calculated")
print(f"\nIngredient score stats:")
print(products_features["ingredient_score"].describe())
print(f"\nPowerhouse products: {products_features['powerhouse_flag'].sum()} out of {len(products_features)}")
print(f"\nTop 10 by ingredient score:")
print(products_features[["product_name", "ingredient_score", "good_ingredient_count", "bad_ingredient_count", "powerhouse_flag"]]
      .dropna(subset=["ingredient_score"])
      .sort_values("ingredient_score", ascending=False)
      .head(10))

Derived ingredient features calculated

Ingredient score stats:
count   1952.000
mean       0.097
std        0.092
min       -0.294
25%        0.050
50%        0.091
75%        0.136
max        1.500
Name: ingredient_score, dtype: float64

Powerhouse products: 1344 out of 1952

Top 10 by ingredient score:
                                           product_name  ingredient_score  \
962                   Powerful-Strength Vitamin C Serum             1.500   
1310                   Salicylic Acid Acne Healing Dots             0.800   
1314                       Salicylic Acid Acne Day Dots             0.800   
35       Beauty Sleep Power Peel Age-Defying Night Peel             0.667   
103                         100% Sugarcane Squalane Oil             0.500   
1868                            100% Niacinamide Powder             0.500   
1864                   Resveratrol 3% + Ferulic Acid 3%             0.500   
1861                        100% L-Ascorbic Acid Powder             0.500   


### Ingredient Scoring Findings
Top 10 by ingredient score:

- Kiehl's Powerful-Strength Vitamin C Serum leads at 1.5 — pure actives, minimal fillers
- Salicylic Acid Acne Dots score 0.8 — targeted single-ingredient products score well
- The Ordinary products dominate the top 10 — makes total sense, they're known for single high-concentration actives with minimal bad ingredients
- 1,344 out of 1,952 products are powerhouse products — that's 69%, which is quite high. We might want to raise the threshold to 4+ ingredients later

## 7. Final Feature Dataset
Combining all engineered features into one clean analytics-ready dataset.
We drop products with no review data since the ML model needs avg_rating 
as its target variable.

In [9]:
# Select all feature columns we've built
feature_cols = [
    # Identity
    "product_id", "product_name", "brand_name", "secondary_category",
    "price_usd", "price_tier", "price_rank_pct",
    
    # Review metrics (target + inputs)
    "avg_rating", "review_count", "five_star_share", "one_star_share",
    "recommendation_rate", "avg_helpfulness",
    
    # Engagement
    "loves_count", "engagement_quality", "review_volume_rank",
    
    # Price value
    "rating_per_dollar",
    
    # Ingredient features
    "ingredient_count", "good_ingredient_count", "bad_ingredient_count",
    "ingredient_score", "good_bad_ratio", "powerhouse_flag", "value_score",
    
    # Product flags
    "limited_edition", "new", "online_only", "sephora_exclusive"
]

# Build final dataset
final_features = products_features[feature_cols].copy()

# Drop products with no review data — ML needs avg_rating as target
final_features_ml = final_features.dropna(subset=["avg_rating"]).copy()

print(f"Full feature dataset: {final_features.shape}")
print(f"ML-ready dataset (has reviews): {final_features_ml.shape}")
print(f"\nNull check on ML dataset:")
print(final_features_ml.isnull().sum()[final_features_ml.isnull().sum() > 0])
print(f"\nSample of final dataset:")
print(final_features_ml.head(3))

Full feature dataset: (1952, 28)
ML-ready dataset (has reviews): (1915, 28)

Null check on ML dataset:
Series([], dtype: int64)

Sample of final dataset:
  product_id                             product_name brand_name  \
0    P439055     GENIUS Sleeping Collagen Moisturizer   Algenist   
1    P421277             GENIUS Liquid Collagen Serum   Algenist   
2    P467602  Triple Algae Eye Renewal Balm Eye Cream   Algenist   

  secondary_category  price_usd price_tier  price_rank_pct  avg_rating  \
0       Moisturizers     98.000    Premium           0.891       4.541   
1         Treatments    115.000    Premium           0.921       4.026   
2           Eye Care     68.000        Mid           0.726       4.531   

   review_count  five_star_share  one_star_share  recommendation_rate  \
0      1321.000            0.706           0.020                0.921   
1      1159.000            0.580           0.109                0.770   
2      1142.000            0.650           0.013         

In [10]:
# Save both versions
import os
os.makedirs("../data/processed", exist_ok=True)

# Full dataset including products with no reviews
final_features.to_csv("../data/processed/features_all.csv", index=False)

# ML ready dataset — only products with review data
final_features_ml.to_csv("../data/processed/features_ml.csv", index=False)

print("=== FILES SAVED ===")
print(f"features_all.csv — {len(final_features)} rows (all skincare products)")
print(f"features_ml.csv — {len(final_features_ml)} rows (products with reviews only)")
print(f"\nColumns saved: {list(final_features.columns)}")

=== FILES SAVED ===
features_all.csv — 1952 rows (all skincare products)
features_ml.csv — 1915 rows (products with reviews only)

Columns saved: ['product_id', 'product_name', 'brand_name', 'secondary_category', 'price_usd', 'price_tier', 'price_rank_pct', 'avg_rating', 'review_count', 'five_star_share', 'one_star_share', 'recommendation_rate', 'avg_helpfulness', 'loves_count', 'engagement_quality', 'review_volume_rank', 'rating_per_dollar', 'ingredient_count', 'good_ingredient_count', 'bad_ingredient_count', 'ingredient_score', 'good_bad_ratio', 'powerhouse_flag', 'value_score', 'limited_edition', 'new', 'online_only', 'sephora_exclusive']


## 8. Feature Summary

Feature engineering is complete. We built 20+ features across 5 groups:

| Group | Features | Purpose |
|---|---|---|
| Review metrics | avg_rating, review_count, five_star_share, one_star_share, recommendation_rate, avg_helpfulness | Measure customer sentiment |
| Price & value | price_rank_pct, rating_per_dollar, price_tier | Put price in context |
| Engagement | loves_count, engagement_quality, review_volume_rank | Measure product popularity |
| Ingredients | ingredient_score, good_bad_ratio, powerhouse_flag, value_score | Measure formula quality |
| Product flags | limited_edition, new, online_only, sephora_exclusive | Capture product characteristics |

**Next step: ML model — train on these features to predict avg_rating, 
then use predicted vs actual gap to identify overrated and underrated products.**